# 🐟 fishaudio/s2-pro — Full Pipeline on TPU v5e-1 / High-RAM

**Runs entirely locally — no remote API calls.**  
Hindi + emotion tags · optional voice cloning · Tony Stark live dashboard.

| Cell | What it does |
|------|-------------|
| **1** | Environment & hardware audit |
| **2** | Configuration (all params here) |
| **3** | Upload `.txt` text file |
| **4** | Upload reference audio (optional, voice cloning) |
| **5** | Install all dependencies |
| **6** | Download `fishaudio/s2-pro` weights (~10 GB) |
| **7** | Run inference — live updating dashboard |
| **8** | Play + download output audio |

> **TPU v5e-1:** Model uses host CPU RAM (~96 GB). `CUDA_VISIBLE_DEVICES=''` hides the TPU HBM from PyTorch so weights load into host RAM. Inference is CPU-only — slower than GPU but the full fp16 model fits without quantization.

In [1]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT & HARDWARE AUDIT
# ════════════════════════════════════════════════════════════
import os, sys, subprocess, platform, time

os.environ['CUDA_VISIBLE_DEVICES']          = ''
os.environ['PJRT_DEVICE']                   = 'CPU'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['OMP_NUM_THREADS']               = '8'
os.environ['MKL_NUM_THREADS']               = '8'

SEP  = "═" * 62
print(f"\n{SEP}")
print("  CELL 1 — ENVIRONMENT & HARDWARE AUDIT")
print(f"{SEP}\n")

print("🐍 PYTHON")
print(f"   Version      : {sys.version.split()[0]}")
print(f"   Executable   : {sys.executable}")
print(f"   Platform     : {platform.platform()}")
print(f"   Architecture : {platform.machine()}")

print(f"\n💻 CPU")
try:
    with open('/proc/cpuinfo') as f:
        cpuinfo = f.read()
    model_lines  = [l for l in cpuinfo.splitlines() if 'model name' in l]
    cpu_model    = model_lines[0].split(':')[1].strip() if model_lines else 'Unknown'
    cpu_logical  = cpuinfo.count('processor\t:')
    print(f"   Model        : {cpu_model}")
    print(f"   Logical cores: {cpu_logical}")
except Exception as e:
    print(f"   /proc/cpuinfo: {e}")

print(f"\n🧠 HOST RAM  ← model loads here")
try:
    with open('/proc/meminfo') as f:
        mi = {l.split(':')[0]: l.split(':')[1].strip() for l in f}
    total_gb = int(mi['MemTotal'].split()[0])     / 1e6
    avail_gb = int(mi['MemAvailable'].split()[0]) / 1e6
    free_gb  = int(mi['MemFree'].split()[0])      / 1e6
    print(f"   Total        : {total_gb:.1f} GB")
    print(f"   Available    : {avail_gb:.1f} GB  ← model needs ~9 GB (fp16)")
    print(f"   Free         : {free_gb:.1f} GB")
    if avail_gb < 12:
        print(f"   ⚠️  Less than 12 GB available — may fail to load model.")
    else:
        print(f"   ✅ Sufficient RAM for full fp16 model.")
except Exception as e:
    print(f"   /proc/meminfo: {e}")
    total_gb = 0

print(f"\n💾 DISK SPACE")
try:
    st = os.statvfs('/')
    disk_total_gb = (st.f_blocks * st.f_frsize) / 1e9
    disk_free_gb  = (st.f_bavail * st.f_frsize) / 1e9
    print(f"   Total        : {disk_total_gb:.1f} GB")
    print(f"   Free         : {disk_free_gb:.1f} GB")
    print(f"   ✅ OK" if disk_free_gb >= 15 else f"   ⚠️  Less than 15 GB free.")
except Exception as e:
    print(f"   statvfs: {e}")

print(f"\n⚡ ACCELERATOR")
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print(f"   torch_xla    : {torch_xla.__version__}")
    print(f"   TPU devices  : {xm.get_xla_supported_devices()}")
    print(f"   ✅ TPU v5e-1 — host RAM ({total_gb:.0f} GB) used for model weights")
except ImportError:
    print(f"   torch_xla    : not found (standard Colab)")

try:
    smi = subprocess.check_output(
        ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],
        stderr=subprocess.DEVNULL).decode().strip()
    print(f"   NVIDIA GPU   : {smi}  ← hidden from PyTorch (CUDA_VISIBLE_DEVICES='')")
except Exception:
    print(f"   NVIDIA GPU   : none / hidden")

print(f"\n🔧 TOOLS")
for tool in ['git','ffmpeg','python3','pip3','curl']:
    try:
        path = subprocess.check_output(['which',tool],stderr=subprocess.DEVNULL).decode().strip()
        print(f"   ✅ {tool:12s} → {path}")
    except:
        print(f"   ❌ {tool:12s} → NOT FOUND")

print(f"\n🔥 PYTORCH")
try:
    import torch
    print(f"   Version      : {torch.__version__}")
    print(f"   CUDA avail   : {torch.cuda.is_available()}  (expected False)")
    torch.set_num_threads(8)
    print(f"   CPU threads  → 8")
except ImportError:
    print(f"   Not installed yet (done in Cell 5)")

print(f"\n{SEP}")
print("  ✅ CELL 1 COMPLETE — Proceed to Cell 2")
print(f"{SEP}")



══════════════════════════════════════════════════════════════
  CELL 1 — ENVIRONMENT & HARDWARE AUDIT
══════════════════════════════════════════════════════════════

🐍 PYTHON
   Version      : 3.12.12
   Executable   : /usr/bin/python3
   Platform     : Linux-6.6.113+-x86_64-with-glibc2.35
   Architecture : x86_64

💻 CPU
   Model        : AMD EPYC 7B13
   Logical cores: 24

🧠 HOST RAM  ← model loads here
   Total        : 49.3 GB
   Available    : 47.6 GB  ← model needs ~9 GB (fp16)
   Free         : 37.4 GB
   ✅ Sufficient RAM for full fp16 model.

💾 DISK SPACE
   Total        : 241.9 GB
   Free         : 221.4 GB
   ✅ OK

⚡ ACCELERATOR
   torch_xla    : 2.9.0
   TPU devices  : ['xla:0']
   ✅ TPU v5e-1 — host RAM (49 GB) used for model weights
   NVIDIA GPU   : none / hidden

🔧 TOOLS
   ✅ git          → /usr/bin/git
   ✅ ffmpeg       → /usr/bin/ffmpeg
   ✅ python3      → /usr/bin/python3
   ✅ pip3         → /usr/local/bin/pip3
   ✅ curl         → /usr/bin/curl

🔥 PYTORCH
   Version 

In [2]:

# ════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ════════════════════════════════════════════════════════════
# Edit variables below. All subsequent cells read from here.
# ════════════════════════════════════════════════════════════
import os

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 2 — CONFIGURATION")
print(f"{SEP}\n")

# ── EDIT THESE ────────────────────────────────────────────────
DO_INSTALL     = True           # False = skip install (if already done)
# MODEL_DIR      = 'checkpoints/s2-pro'
# NEW (absolute — works regardless of CWD)
MODEL_DIR = '/content/checkpoints/s2-pro'
OUTPUT_DIR = '/content/inference_outputs'
HF_TOKEN       = ''             # Leave empty for public model access

PROMPT_TEXT    = ''             # Exact transcript of reference audio (for voice cloning)

TEMPERATURE    = 0.5            # 0.1=robotic  0.5=stable+natural  1.0=expressive
TOP_P          = 0.8            # Nucleus sampling (0.1–1.0)
TOP_K          = 50             # Top-K vocab filter
MAX_NEW_TOKENS = 2048           # 0 = model decides. Raise for long texts.
COMPILE        = False          # KEEP False on Colab/TPU

# ── VOICE SAMPLING ───────────────────────────────────────────
# NUM_SAMPLES: how many random voice takes to generate.
# Without reference audio, each take picks a DIFFERENT random voice.
# Use 2-3 to audition voices; each take ~adds equal generation time.
# ⚠️  Tags like [young female voice] control PROSODY/STYLE only —
#     NOT speaker identity. For a consistent Indian female voice
#     you MUST upload a ~10s reference audio clip in Cell 4.
NUM_SAMPLES    = 3              # 1=fastest, 3=audition 3 random voices; keep best as reference

# ── AUDIOBOOK / NARRATION MODE ──────────────────────────────
# AUDIOBOOK_MODE = True enables smart Hinglish text normalizer:
#   • English abbreviations/acronyms → phonetic forms (S2→S-Two, AI→ए.आई)
#   • Injects NARRATOR_TAG as the single voice descriptor
#   • Strips tag-clutter from input text (over-tagging hurts quality)
# Set False if you want to pass raw tagged text without any processing.
AUDIOBOOK_MODE = True

# NARRATOR_TAG — ONE natural-language description of your narrator.
# This is the MOST powerful lever for voice in S2-Pro (free-form text).
# Keep it in English for best effect (model trained primarily on EN tags).
# It will be prepended to your text automatically when AUDIOBOOK_MODE=True.
# Examples:
#   casual Indian girl: 'young Indian woman, casual conversational Hinglish,
#                        warm friendly tone, clear natural Hindi-English mix'
#   pro audiobook:      'professional Indian female narrator, crisp diction,
#                        moderate storytelling pace, Indian English accent'
#   millennial podcaster:'millennial Indian woman, energetic podcast host voice,
#                         natural Hinglish, urban Indian accent'
NARRATOR_TAG = "young Indian woman, casual conversational Hinglish, warm and clear, natural Hindi-English mix, moderate pace, audiobook narration, urban Indian accent"

# OUTPUT_DIR already set above as absolute path
# ─────────────────────────────────────────────────────────────

os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs('uploads',  exist_ok=True)

if HF_TOKEN.strip():
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    os.environ['HF_TOKEN']              = HF_TOKEN
    print(f"   ✅ HF_TOKEN applied  (length: {len(HF_TOKEN)})")
else:
    print(f"   ℹ️  No HF_TOKEN — public model access (usually fine)")

print(f"\n{'─'*40}")
print(f"   DO_INSTALL     : {DO_INSTALL}")
print(f"   MODEL_DIR      : {MODEL_DIR}/")
print(f"   OUTPUT_DIR     : {OUTPUT_DIR}/")
print(f"   TEMPERATURE    : {TEMPERATURE}")
print(f"   TOP_P          : {TOP_P}")
print(f"   TOP_K          : {TOP_K}")
print(f"   MAX_NEW_TOKENS : {MAX_NEW_TOKENS}  (0 = auto)")
print(f"   COMPILE        : {COMPILE}")
print(f"   NUM_SAMPLES    : {NUM_SAMPLES}")
print(f"   AUDIOBOOK_MODE : {AUDIOBOOK_MODE}")
print(f"   NARRATOR_TAG   : {NARRATOR_TAG[:60]}...")
print(f"   PROMPT_TEXT    : {'एक डॉलर सत्तासी सेंट। बस इतना ही था। और उसमें से साठ सेंट तो पैनीज़ में थे। पैनीज़ जो उन्होंने ग्रोसर आंटी और सब्जी वाले अंकल को तंग करते हुए बचाए थे... जब तक उनके गाल लाल नहीं हो जाते थे। लोग सोचते थे कि वह बहुत कंजूस हैं। “डेला,” वह तीन बार गिन ली — एक डॉलर सत्तासी सेंट... “और कल क्रिसमस है...!!!” स्पष्ट था कि कुछ बड़ा करने का तो रास्ता ही नहीं था... बस पुराने गंदे सोफे पर गिर कर रोना ही सही रहेगा। तो डेला रो पड़ी।'}")
print(f"{'─'*40}")

# Initialise so later cells never hit NameError even if skipped
TEXT_TO_SYNTH   = "नमस्ते! यह FishAudio S2-Pro का परीक्षण है।"  # fallback
REFERENCE_AUDIO = ''   # ← will be set by Cell 4; empty = random voice

print(f"\n   TEXT_TO_SYNTH   : set to fallback (overwritten by Cell 3)")
print(f"   REFERENCE_AUDIO : '' (overwritten by Cell 4 if you upload audio)")

print(f"\n{SEP}")
print("  ✅ CELL 2 COMPLETE — Proceed to Cell 3")
print(f"{SEP}")



══════════════════════════════════════════════════════════════
  CELL 2 — CONFIGURATION
══════════════════════════════════════════════════════════════

   ℹ️  No HF_TOKEN — public model access (usually fine)

────────────────────────────────────────
   DO_INSTALL     : True
   MODEL_DIR      : /content/checkpoints/s2-pro/
   OUTPUT_DIR     : /content/inference_outputs/
   TEMPERATURE    : 0.5
   TOP_P          : 0.8
   TOP_K          : 50
   MAX_NEW_TOKENS : 2048  (0 = auto)
   COMPILE        : False
   NUM_SAMPLES    : 3
   AUDIOBOOK_MODE : True
   NARRATOR_TAG   : young Indian woman, casual conversational Hinglish, warm and...
   PROMPT_TEXT    : एक डॉलर सत्तासी सेंट। बस इतना ही था। और उसमें से साठ सेंट तो पैनीज़ में थे। पैनीज़ जो उन्होंने ग्रोसर आंटी और सब्जी वाले अंकल को तंग करते हुए बचाए थे... जब तक उनके गाल लाल नहीं हो जाते थे। लोग सोचते थे कि वह बहुत कंजूस हैं। “डेला,” वह तीन बार गिन ली — एक डॉलर सत्तासी सेंट... “और कल क्रिसमस है...!!!” स्पष्ट था कि कुछ बड़ा करने का तो रास्ता ह

In [3]:
# ════════════════════════════════════════════════════════════
# CELL 3 — UPLOAD TEXT FILE  (.txt)
# ════════════════════════════════════════════════════════════
import os
from google.colab import files

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 3 — UPLOAD TEXT FILE")
print(f"{SEP}")
print("""
  Accepted : .txt  (UTF-8 encoding)
  Emotion tags supported:
    [excited]  [whisper]  [pause]   [laugh]   [sad]
    [singing]  [shouting] [emphasis][sigh]    [fast pace]
    [chuckle]  [inhale]   [volume up/down]
  Example:
    [excited] नमस्ते भाई! [pause] आज बहुत मज़ा आएगा।
  Tip: keep under ~500 words for first test.
""")

print("📂 File picker opening — select your .txt file...")
uploaded_text = files.upload()

if uploaded_text:
    fname = list(uploaded_text.keys())[0]
    raw   = uploaded_text[fname]

    dest = os.path.join('uploads', fname)
    with open(dest, 'wb') as f:
        f.write(raw)

    try:
        TEXT_TO_SYNTH = raw.decode('utf-8').strip()
    except UnicodeDecodeError:
        TEXT_TO_SYNTH = raw.decode('utf-8', errors='replace').strip()

    word_count = len(TEXT_TO_SYNTH.split())
    char_count = len(TEXT_TO_SYNTH)
    line_count = TEXT_TO_SYNTH.count('\n') + 1

    print(f"\n✅ FILE LOADED")
    print(f"{'─'*50}")
    print(f"   Filename   : {fname}")
    print(f"   Saved to   : {dest}")
    print(f"   Bytes      : {len(raw):,}")
    print(f"   Characters : {char_count:,}")
    print(f"   Words      : {word_count:,}")
    print(f"   Lines      : {line_count:,}")
    print(f"{'─'*50}")

    if word_count > 500:
        est_min = word_count / 30   # CPU ~30 words/min rough estimate
        print(f"   ⚠️  {word_count} words — estimated CPU time: ~{est_min:.0f} min")
        print(f"      Consider splitting into smaller chunks first.")
    elif word_count > 150:
        print(f"   ℹ️  {word_count} words — moderate length, may take a few minutes on CPU")
    else:
        print(f"   ✅ Good length for first test.")

    print(f"\n📖 PREVIEW (first 500 chars):")
    print(f"{'─'*50}")
    print(TEXT_TO_SYNTH[:500] + ('...' if char_count > 500 else ''))
    print(f"{'─'*50}")
else:
    TEXT_TO_SYNTH = "नमस्ते! यह FishAudio S2-Pro का परीक्षण है। कृपया इसे हिंदी में बोलिए।"
    print("\n⚠️  No file uploaded — using default Hindi text.")
    print(f"   Text: {TEXT_TO_SYNTH}")

print(f"\n   TEXT_TO_SYNTH set → {len(TEXT_TO_SYNTH)} chars / {len(TEXT_TO_SYNTH.split())} words")
print(f"\n{SEP}")
print("  ✅ CELL 3 COMPLETE — Proceed to Cell 4")
print(f"{SEP}")



══════════════════════════════════════════════════════════════
  CELL 3 — UPLOAD TEXT FILE
══════════════════════════════════════════════════════════════

  Accepted : .txt  (UTF-8 encoding)
  Emotion tags supported:
    [excited]  [whisper]  [pause]   [laugh]   [sad]
    [singing]  [shouting] [emphasis][sigh]    [fast pace]
    [chuckle]  [inhale]   [volume up/down]
  Example:
    [excited] नमस्ते भाई! [pause] आज बहुत मज़ा आएगा।
  Tip: keep under ~500 words for first test.

📂 File picker opening — select your .txt file...


Saving four_words.txt to four_words.txt

✅ FILE LOADED
──────────────────────────────────────────────────
   Filename   : four_words.txt
   Saved to   : uploads/four_words.txt
   Bytes      : 217
   Characters : 142
   Words      : 21
   Lines      : 2
──────────────────────────────────────────────────
   ✅ Good length for first test.

📖 PREVIEW (first 500 chars):
──────────────────────────────────────────────────
[female conversational Hinglish] अरे यार, यह S-Two model सुनो — A-I कमाल है! [emphasis] बिल्कुल real लगता है! [shocked] कमाल! 
Unbelievable!!!
──────────────────────────────────────────────────

   TEXT_TO_SYNTH set → 142 chars / 21 words

══════════════════════════════════════════════════════════════
  ✅ CELL 3 COMPLETE — Proceed to Cell 4
══════════════════════════════════════════════════════════════


In [7]:
# ════════════════════════════════════════════════════════════
# CELL 4 — UPLOAD REFERENCE AUDIO  (OPTIONAL — voice cloning)
# ════════════════════════════════════════════════════════════
# THIS CELL WAS MISSING — it defines REFERENCE_AUDIO which
# Cell 7 needs. Without it, Cell 7 raises NameError.
#
# Upload a short .wav/.mp3 to clone that voice.
# If you skip/cancel, REFERENCE_AUDIO stays '' and the model
# picks a random high-quality built-in voice.
#
# For best voice cloning:
#   ✅ 3–15 seconds of clean speech
#   ✅ No music, reverb, or background noise
#   ✅ Also fill PROMPT_TEXT in Cell 2 with its exact transcript
#   ❌ Clips >30s not recommended
# ════════════════════════════════════════════════════════════
import os, json
from google.colab import files

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 4 — UPLOAD REFERENCE AUDIO  (OPTIONAL)")
print(f"{SEP}")
print("""
  ┌─────────────────────────────────────────────────────┐
  │  OPTIONAL — skip this if you want a random voice.  │
  │  Click Cancel or don't upload to use random voice. │
  └─────────────────────────────────────────────────────┘

  If uploading audio:
    1. Upload .wav or .mp3 here
    2. Set PROMPT_TEXT in Cell 2 = exact transcript of clip

  Without PROMPT_TEXT, voice cloning quality drops.
""")

# Always define REFERENCE_AUDIO with a safe default
# This is the variable that Cell 7 uses — it MUST be defined.
REFERENCE_AUDIO      = ''
REFERENCE_AUDIO_NAME = ''
# PROMPT_TEXT is intentionally NOT set here.
# It is inherited from Cell 2 where the user configured it.
# (The previous hardcoded Hindi sample here was silently overwriting Cell 2's setting.)
# If PROMPT_TEXT was not set in Cell 2, default to empty string.
if 'PROMPT_TEXT' not in dir():
    PROMPT_TEXT = ''

print("📂 File picker — upload reference audio or cancel to skip...")
try:
    uploaded_audio = files.upload()
except Exception:
    uploaded_audio = {}

if uploaded_audio:
    aname = list(uploaded_audio.keys())[0]
    raw   = uploaded_audio[aname]
    ext   = os.path.splitext(aname)[1].lower()

    SUPPORTED = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac')
    if ext not in SUPPORTED:
        print(f"\n   ⚠️  Extension '{ext}' not in {SUPPORTED}")
        print(f"   Continuing anyway — ffmpeg may still handle it.")

    # Use absolute path so Cell 7 can find the file regardless of CWD
    os.makedirs('/content/uploads', exist_ok=True)
    dest = os.path.join('/content/uploads', aname)
    with open(dest, 'wb') as f:
        f.write(raw)

    REFERENCE_AUDIO      = dest
    REFERENCE_AUDIO_NAME = aname
    size_kb = len(raw) / 1024

    print(f"\n✅ REFERENCE AUDIO LOADED")
    print(f"{'─'*50}")
    print(f"   Filename    : {aname}")
    print(f"   Saved to    : {dest}")
    print(f"   Size        : {size_kb:.1f} KB  ({len(raw):,} bytes)")
    print(f"   Format      : {ext}")
    print(f"{'─'*50}")

    # Probe with ffprobe
    try:
        import subprocess as _sp
        r = _sp.run(
            ['ffprobe','-v','quiet','-print_format','json','-show_streams', dest],
            capture_output=True, text=True, timeout=10
        )
        probe = json.loads(r.stdout)
        st = probe['streams'][0]
        duration    = float(st.get('duration', 0))
        sample_rate = st.get('sample_rate', '?')
        channels_n  = st.get('channels', '?')
        codec       = st.get('codec_name', '?')
        print(f"   Duration    : {duration:.2f} s")
        print(f"   Sample rate : {sample_rate} Hz")
        print(f"   Channels    : {channels_n}")
        print(f"   Codec       : {codec}")
        if duration < 2:
            print(f"   ⚠️  Very short (<2s) — quality may suffer.")
        elif duration > 30:
            print(f"   ⚠️  Long clip (>30s) — first ~15s will be used.")
        else:
            print(f"   ✅ Good clip length for voice cloning.")
    except Exception as e:
        print(f"   (Could not probe audio: {e})")


    # Warn if transcript missing
    if not PROMPT_TEXT.strip():
        print(f"\n   ⚠️  PROMPT_TEXT is empty in Cell 2!")
        print(f"   Voice cloning quality is much better with a transcript.")
        print(f"   → Go back to Cell 2 and set PROMPT_TEXT to what is spoken.")
    else:
        excerpt = PROMPT_TEXT[:80] + ('...' if len(PROMPT_TEXT) > 80 else '')
        print(f"\n   PROMPT_TEXT : '{excerpt}'")
        print(f"   ✅ Transcript provided — full voice cloning enabled.")

else:
    # No upload — random voice
    REFERENCE_AUDIO = ''
    print("\n✅ No reference audio — will use random built-in voice.")
    print("   S2-Pro's default voices are expressive and multilingual.")

print(f"\n{'─'*50}")
print(f"   REFERENCE_AUDIO = '{REFERENCE_AUDIO}'")
mode = f"VOICE CLONE: {REFERENCE_AUDIO_NAME}" if REFERENCE_AUDIO else "RANDOM VOICE (no reference)"
print(f"   Mode            = {mode}")
print(f"{'─'*50}")

print(f"\n{SEP}")
print("  ✅ CELL 4 COMPLETE — Proceed to Cell 5")
print(f"{SEP}")



══════════════════════════════════════════════════════════════
  CELL 4 — UPLOAD REFERENCE AUDIO  (OPTIONAL)
══════════════════════════════════════════════════════════════

  ┌─────────────────────────────────────────────────────┐
  │  OPTIONAL — skip this if you want a random voice.  │
  │  Click Cancel or don't upload to use random voice. │
  └─────────────────────────────────────────────────────┘

  If uploading audio:
    1. Upload .wav or .mp3 here
    2. Set PROMPT_TEXT in Cell 2 = exact transcript of clip

  Without PROMPT_TEXT, voice cloning quality drops.

📂 File picker — upload reference audio or cancel to skip...


Saving Fiction___1.0x.mp3 to Fiction___1.0x.mp3

✅ REFERENCE AUDIO LOADED
──────────────────────────────────────────────────
   Filename    : Fiction___1.0x.mp3
   Saved to    : uploads/Fiction___1.0x.mp3
   Size        : 141.9 KB  (145,293 bytes)
   Format      : .mp3
──────────────────────────────────────────────────
   Duration    : 36.26 s
   Sample rate : 24000 Hz
   Channels    : 1
   Codec       : mp3
   ⚠️  Long clip (>30s) — first ~15s will be used.

   PROMPT_TEXT : 'एक डॉलर सत्तासी सेंट।
                  बस इतना ही था।
                  और उसमे...'
   ✅ Transcript provided — full voice cloning enabled.

──────────────────────────────────────────────────
   REFERENCE_AUDIO = 'uploads/Fiction___1.0x.mp3'
   Mode            = VOICE CLONE: Fiction___1.0x.mp3
──────────────────────────────────────────────────

══════════════════════════════════════════════════════════════
  ✅ CELL 4 COMPLETE — Proceed to Cell 5
══════════════════════════════════════════════════════════════


In [8]:
# ════════════════════════════════════════════════════════════
# CELL 5 — INSTALL DEPENDENCIES
# ════════════════════════════════════════════════════════════
# Install order matters:
#  1. apt system packages
#  2. pip upgrade
#  3. PyTorch CPU build  (no CUDA overhead for host-RAM mode)
#  4. transformers FIRST → pip auto-picks huggingface-hub <1.0
#  5. fish-speech specific packages
#  6. fish-speech repo clone + editable install
# ════════════════════════════════════════════════════════════
import os, sys, subprocess, time

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 5 — INSTALLING DEPENDENCIES")
print(f"{SEP}")

if not DO_INSTALL:
    print("\n  DO_INSTALL = False — skipping all installations.")
    print(f"\n{SEP}")
    print("  ✅ CELL 5 SKIPPED")
    print(f"{SEP}")
    raise SystemExit(0)

def banner(s):
    print(f"\n{'─'*60}\n  📦 {s}\n{'─'*60}")

def pip_install(label, pkgs, no_deps=False, allow_fail=False, extra_index=None):
    t0 = time.time()
    print(f"   ▶ {label}...", end='', flush=True)
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location']
    if no_deps:
        cmd.append('--no-deps')
    if extra_index:
        cmd += ['--index-url', extra_index]
    cmd += (pkgs if isinstance(pkgs, list) else [pkgs])
    r = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    if r.returncode == 0:
        print(f"  ✅  ({elapsed:.1f}s)")
    else:
        if allow_fail:
            last = [l for l in r.stderr.strip().splitlines() if l.strip()]
            msg = last[-1][:100] if last else '(no stderr)'
            print(f"  ⚠️  non-fatal ({elapsed:.1f}s)\n     {msg}")
        else:
            print(f"  ❌  ({elapsed:.1f}s)")
            print(r.stderr.strip()[-400:])
            raise RuntimeError(f"pip install failed: {label}")
    return r.returncode == 0

grand_t0 = time.time()

# ── 1: System packages ───────────────────────────────────────
banner("1 / 6 — System packages (apt)")
subprocess.run(['apt-get','update','-qq'], capture_output=True)
subprocess.run(['apt-get','install','-y','-qq',
                'ffmpeg','libsox-dev','sox','libsndfile1-dev',
                'libasound2-dev','cmake','build-essential','git'],
               capture_output=True)
print("   ✅  apt packages installed")

# ── 2: Pip upgrade ───────────────────────────────────────────
banner("2 / 6 — Upgrade pip, setuptools, wheel")
pip_install("pip upgrade", ['pip','setuptools','wheel','--upgrade'])

# ── 3: PyTorch CPU ───────────────────────────────────────────
banner("3 / 6 — PyTorch CPU build")
print("   Installing torch + torchaudio (CPU, no CUDA) — ~300 MB, 2-4 min...")
t0 = time.time()
r = subprocess.run(
    [sys.executable,'-m','pip','install','torch','torchaudio',
     '--index-url','https://download.pytorch.org/whl/cpu',
     '-q','--no-warn-script-location'],
    capture_output=True, text=True
)
if r.returncode == 0:
    import torch
    torch.set_num_threads(8)
    print(f"   ✅  torch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  threads: {torch.get_num_threads()}  ({time.time()-t0:.0f}s)")
else:
    print(f"   ⚠️  CPU wheel failed, trying default torch...")
    subprocess.run([sys.executable,'-m','pip','install','torch','torchaudio','-q'],
                   capture_output=True)

# ── 4: transformers FIRST (huggingface-hub <1.0 auto-picked) ─
banner("4 / 6 — transformers + core packages (install ORDER matters)")
print("   Installing transformers first → pip picks huggingface-hub <1.0 automatically")
print("   (If hf_hub is installed first it resolves to 1.6.0 and breaks transformers)")
for pkg, lbl in [
    ('transformers>=4.45.2,<=4.57.3',  'transformers (pins hf_hub<1.0)'),
    ('accelerate>=0.26.0',              'accelerate'),
    ('vector_quantize_pytorch==1.14.24','vector_quantize_pytorch'),
    ('numpy>=1.26.0',                   'numpy'),
    ('scipy',                           'scipy'),
    ('librosa>=0.10.1',                'librosa'),
    ('soundfile',                       'soundfile'),
]:
    pip_install(lbl, pkg)

# ── 5: fish-speech specific ──────────────────────────────────
banner("5 / 6 — fish-speech specific packages")
FISH_PKGS = [
    ('loguru>=0.6.0',               'loguru'),
    ('einops>=0.7.0',               'einops'),
    ('loralib>=0.1.2',              'loralib'),
    ('pyrootutils>=1.0.4',          'pyrootutils'),
    ('hydra-core>=1.3.2',           'hydra-core'),
    ('lightning>=2.1.0',            'lightning'),
    ('natsort>=8.4.0',              'natsort'),
    ('rich>=13.5.3',                'rich'),
    ('tiktoken>=0.8.0',             'tiktoken'),
    ('pydantic>=2.9.2',             'pydantic'),
    ('zstandard>=0.22.0',           'zstandard'),
    ('safetensors',                 'safetensors'),
    ('einx[torch]==0.2.2',          'einx'),
    ('resampy>=0.4.3',              'resampy'),
    ('pydub',                       'pydub'),
    ('tqdm',                        'tqdm'),
    ('silero-vad',                  'silero-vad'),
    ('opencc-python-reimplemented==0.1.7', 'opencc'),
    ('datasets==2.18.0',            'datasets'),
    ('rotary-embedding-torch',      'rotary-embedding-torch'),
    ('descript-audio-codec',        'descript-audio-codec'),
]
for pkg, lbl in FISH_PKGS:
    pip_install(lbl, pkg, allow_fail=True)

# ── 6: fish-speech repo ──────────────────────────────────────
banner("6 / 6 — fish-speech GitHub repo + editable install")
REPO_DIR = '/content/fish-speech'
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print("   Cloning fishaudio/fish-speech...", end='', flush=True)
    t0 = time.time()
    r = subprocess.run(
        ['git','clone','--depth','1',
         'https://github.com/fishaudio/fish-speech.git', REPO_DIR],
        capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  ✅  ({time.time()-t0:.0f}s)")
    else:
        print(f"  ❌\n{r.stderr[-200:]}")
else:
    print(f"   ✅  {REPO_DIR} already exists")

if os.path.exists(REPO_DIR):
    pip_install("fish-speech editable", ['-e', REPO_DIR, '--no-deps'],
                no_deps=False, allow_fail=True)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print(f"   ✅  {REPO_DIR} added to sys.path")

# ── Verify ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("  🔍 IMPORT VERIFICATION (fresh subprocess per package)")
print(f"{'─'*60}")
VERIFY = [
    ('torch',                   'import torch; print(torch.__version__)'),
    ('torchaudio',              'import torchaudio; print(torchaudio.__version__)'),
    ('transformers',            'import transformers; print(transformers.__version__)'),
    ('huggingface_hub',         'import huggingface_hub; print(huggingface_hub.__version__)'),
    ('vector_quantize_pytorch', 'import vector_quantize_pytorch; print("ok")'),
    ('loguru',                  'import loguru; print(loguru.__version__)'),
    ('soundfile',               'import soundfile; print(soundfile.__version__)'),
    ('tiktoken',                'import tiktoken; print(tiktoken.__version__)'),
    ('loralib',                 'import loralib; print("ok")'),
    ('fish_speech',             'import fish_speech; print("ok")'),
]
passed, failed = [], []
for name, code in VERIFY:
    r = subprocess.run([sys.executable,'-c',code], capture_output=True, text=True)
    if r.returncode == 0:
        ver = r.stdout.strip().splitlines()[-1]
        print(f"   ✅  {name:35s} {ver}")
        passed.append(name)
    else:
        err = (r.stderr + r.stdout).strip()[-80:]
        print(f"   ❌  {name:35s} {err}")
        failed.append(name)

total_t = time.time() - grand_t0
print(f"\n   Total time : {total_t:.0f}s  ({total_t/60:.1f} min)")
print(f"   Passed     : {len(passed)}/{len(VERIFY)}")
if failed:
    print(f"   Failed     : {failed}")
    print(f"\n   ⚠️  Try: Runtime → Restart session, then re-run Cell 5.")
else:
    print(f"   ✅ All imports verified!")

print(f"\n{SEP}")
print("  ✅ CELL 5 COMPLETE — Proceed to Cell 6")
print(f"  ⚠️  If any import failed: Runtime → Restart → re-run Cell 5")
print(f"{SEP}")



══════════════════════════════════════════════════════════════
  CELL 5 — INSTALLING DEPENDENCIES
══════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  📦 1 / 6 — System packages (apt)
────────────────────────────────────────────────────────────
   ✅  apt packages installed

────────────────────────────────────────────────────────────
  📦 2 / 6 — Upgrade pip, setuptools, wheel
────────────────────────────────────────────────────────────
   ▶ pip upgrade...  ✅  (7.6s)

────────────────────────────────────────────────────────────
  📦 3 / 6 — PyTorch CPU build
────────────────────────────────────────────────────────────
   Installing torch + torchaudio (CPU, no CUDA) — ~300 MB, 2-4 min...
   ✅  torch 2.9.0+cpu  |  CUDA: False  |  threads: 8  (1s)

────────────────────────────────────────────────────────────
  📦 4 / 6 — transformers + core packages (install ORDER matters)
──────────────────────────────────────────────

In [9]:
# ════════════════════════════════════════════════════════════
# CELL 6 — DOWNLOAD MODEL WEIGHTS  (fishaudio/s2-pro)
# ════════════════════════════════════════════════════════════
# Uses huggingface_hub Python API — NOT the CLI.
# The CLI (huggingface-cli) breaks after Runtime Restart
# because shell PATH resets. Python API always works.
# Shows tqdm progress. Resumes partial downloads automatically.
# ════════════════════════════════════════════════════════════
import os, sys, time

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 6 — DOWNLOADING MODEL WEIGHTS")
print(f"{SEP}")
print(f"""
   Model      : fishaudio/s2-pro
   Destination: {MODEL_DIR}
   Size       : ~10 GB  (first run)
   Resume     : ✅ existing files are skipped automatically
   Method     : huggingface_hub Python API  (not CLI)
""")

try:
    import huggingface_hub as hfhub
    print(f"   huggingface_hub : {hfhub.__version__} ✅")
except ImportError:
    print("   huggingface_hub not found — installing...")
    import subprocess
    subprocess.run([sys.executable,'-m','pip','install','huggingface_hub','-q'],
                   capture_output=True)
    import huggingface_hub as hfhub

from huggingface_hub import snapshot_download, hf_hub_download

# ── Show existing files ─────────────────────────────────────
print(f"\n🔍 Scanning {MODEL_DIR} ...")
existing = []
if os.path.exists(MODEL_DIR):
    for root, dirs, fnames in os.walk(MODEL_DIR):
        for fn in sorted(fnames):
            fp  = os.path.join(root, fn)
            rel = os.path.relpath(fp, MODEL_DIR)
            sz  = os.path.getsize(fp)
            existing.append((rel, sz))

if existing:
    total_cached = sum(s for _, s in existing)
    print(f"   Found {len(existing)} cached file(s)  ({total_cached/1e9:.2f} GB total):")
    for rel, sz in sorted(existing):
        bar_len = int(sz / 1e8)  # 1 block per 100 MB
        bar = '█' * min(bar_len, 20)
        print(f"   📄 {rel:50s} {sz/1e6:8.1f} MB  {bar}")
    REQUIRED = ['config.json', 'codec.pth']
    have_req  = all(any(r.endswith(req) for r, _ in existing) for req in REQUIRED)
    if have_req:
        print(f"\n   ✅ Critical files (config.json + codec.pth) already cached.")
        print(f"   snapshot_download will skip them automatically.")
    else:
        missing = [r for r in REQUIRED if not any(e.endswith(r) for e, _ in existing)]
        print(f"\n   ⚠️  Missing critical files: {missing}")
else:
    print(f"   📭 Empty — full download needed (~10 GB, 5–25 min)")

# ── Download ────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  📥 Running snapshot_download() ...")
print(f"  tqdm progress bars appear below for each file")
print(f"{'─'*60}\n")

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

t0 = time.time()
try:
    local_path = snapshot_download(
        repo_id                = 'fishaudio/s2-pro',
        repo_type              = 'model',
        local_dir              = MODEL_DIR,
        local_dir_use_symlinks = False,
        token                  = HF_TOKEN.strip() or None,
        ignore_patterns        = ['*.msgpack','flax_model*','tf_model*','rust_model*'],
    )
    elapsed = time.time() - t0
    print(f"\n   ✅ snapshot_download complete  ({elapsed/60:.1f} min)")
    print(f"   Files at: {local_path}")

except Exception as e:
    print(f"\n   ⚠️  snapshot_download raised {type(e).__name__}: {e}")
    print("   Trying file-by-file fallback ...")
    CRITICAL = ['config.json','codec.pth','model.pth',
                'tokenizer.json','tokenizer_config.json']
    any_ok = False
    for fname in CRITICAL:
        dest = os.path.join(MODEL_DIR, fname)
        if os.path.exists(dest):
            print(f"   ✅ {fname} — already present, skip")
            continue
        try:
            print(f"   📥 {fname} ...", end='', flush=True)
            hf_hub_download(repo_id='fishaudio/s2-pro', filename=fname,
                            local_dir=MODEL_DIR, local_dir_use_symlinks=False,
                            token=HF_TOKEN.strip() or None)
            sz = os.path.getsize(dest) / 1e6
            print(f" ✅  ({sz:.1f} MB)")
            any_ok = True
        except Exception as fe:
            print(f" ⚠️  {type(fe).__name__}")
    if not any_ok:
        raise RuntimeError("All download attempts failed. Check network / HF status.")

# ── Final inventory ─────────────────────────────────────────
print(f"\n{'─'*60}")
print("  🔍 FINAL INVENTORY")
print(f"{'─'*60}")
total_gb = 0.0
all_files = []
for root, dirs, fnames in os.walk(MODEL_DIR):
    for fn in sorted(fnames):
        fp  = os.path.join(root, fn)
        rel = os.path.relpath(fp, MODEL_DIR)
        sz  = os.path.getsize(fp)
        total_gb += sz / 1e9
        all_files.append((rel, sz))
        print(f"   📄 {rel:50s} {sz/1e6:8.1f} MB")
print(f"   {'─'*55}")
print(f"   {'Total:':50s} {total_gb:8.2f} GB")

REQUIRED = ['config.json', 'codec.pth']
print(f"\n  🔍 CRITICAL FILE CHECK")
all_ok = True
for req in REQUIRED:
    found = any(r.endswith(req) for r, _ in all_files)
    print(f"   {'✅' if found else '❌ MISSING'}  {req}")
    if not found: all_ok = False

if not all_ok:
    raise RuntimeError("Critical files missing. Re-run this cell to resume download.")

print(f"\n{SEP}")
print("  ✅ CELL 6 COMPLETE — Proceed to Cell 7")
print(f"{SEP}")



══════════════════════════════════════════════════════════════
  CELL 6 — DOWNLOADING MODEL WEIGHTS
══════════════════════════════════════════════════════════════

   Model      : fishaudio/s2-pro
   Destination: /content/checkpoints/s2-pro
   Size       : ~10 GB  (first run)
   Resume     : ✅ existing files are skipped automatically
   Method     : huggingface_hub Python API  (not CLI)

   huggingface_hub : 0.36.2 ✅

🔍 Scanning /content/checkpoints/s2-pro ...
   📭 Empty — full download needed (~10 GB, 5–25 min)

────────────────────────────────────────────────────────────
  📥 Running snapshot_download() ...
  tqdm progress bars appear below for each file
────────────────────────────────────────────────────────────



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

overview.png:   0%|          | 0.00/3.54M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/12.2M [00:00<?, ?B/s]


   ✅ snapshot_download complete  (1.5 min)
   Files at: /content/checkpoints/s2-pro

────────────────────────────────────────────────────────────
  🔍 FINAL INVENTORY
────────────────────────────────────────────────────────────
   📄 .gitattributes                                          0.0 MB
   📄 LICENSE.md                                              0.0 MB
   📄 README.md                                               0.0 MB
   📄 chat_template.jinja                                     0.0 MB
   📄 codec.pth                                            1871.1 MB
   📄 config.json                                             0.0 MB
   📄 model-00001-of-00002.safetensors                     4986.9 MB
   📄 model-00002-of-00002.safetensors                     4136.9 MB
   📄 model.safetensors.index.json                            0.0 MB
   📄 overview.png                                            3.5 MB
   📄 special_tokens_map.json                                 0.1 MB
   📄 tokenizer.json     

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — S2-PRO INFERENCE  (Tony Stark Live Dashboard)
# ════════════════════════════════════════════════════════════
# 3-stage pipeline:
#   Stage 1 (optional): DAC encode reference audio → .npy
#   Stage 2           : text2semantic LLM → codes_*.npy
#   Stage 3           : DAC decode codes → output.wav
#
# Live dashboard — real token counts parsed from subprocess output:
#   Parses "128/512 [00:30<01:30, 1.42it/s]"  (tqdm full)
#         "128/512"                             (bare fraction)
#         "generate 512 tokens"                (fish-speech log)
#         "tokens: 512"                        (key=value)
#   ► stage_pct  = tok_done / tok_total * 100  (never fake-capped)
#   ► overall_pct follows real stage_pct
#   ► speed shown to 2 d.p., ETA from tqdm's own timer
# ════════════════════════════════════════════════════════════
import os, sys, subprocess, glob, time, gc, re, shlex
import html as _html
from IPython.display import display, HTML

REPO_DIR = '/content/fish-speech'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 7 — S2-PRO INFERENCE")
print(f"{SEP}\n")

# ── Pre-flight ───────────────────────────────────────────────
print("🔍 PRE-FLIGHT CHECKS")
print(f"   TEXT_TO_SYNTH   : {len(TEXT_TO_SYNTH)} chars / {len(TEXT_TO_SYNTH.split())} words")
print(f"   REFERENCE_AUDIO : '{REFERENCE_AUDIO}'" if REFERENCE_AUDIO
      else "   REFERENCE_AUDIO : '' (random voice)")
print(f"   MODEL_DIR       : {MODEL_DIR}")
print(f"   OUTPUT_DIR      : {OUTPUT_DIR}")
print(f"   TEMPERATURE     : {TEMPERATURE}")
print(f"   TOP_P / TOP_K   : {TOP_P} / {TOP_K}")
print(f"   COMPILE         : {COMPILE}")

for req in ['config.json', 'codec.pth']:
    rp = os.path.join(MODEL_DIR, req)
    if not os.path.exists(rp):
        raise FileNotFoundError(f"Missing: {rp} — re-run Cell 6.")
    print(f"   ✅ {req}")

if not TEXT_TO_SYNTH.strip():
    raise ValueError("TEXT_TO_SYNTH is empty — run Cell 3 first.")
print(f"   ✅ Input text present")

try:
    with open('/proc/meminfo') as f:
        mi = {l.split(':')[0]: l.split(':')[1].strip() for l in f}
    _avail_gb = int(mi['MemAvailable'].split()[0]) / 1e6
    _total_gb = int(mi['MemTotal'].split()[0])     / 1e6
    print(f"   RAM available   : {_avail_gb:.1f} GB / {_total_gb:.1f} GB")
    if _avail_gb < 10:
        print(f"   ⚠️  Less than 10 GB free — model load may be slow/fail.")
except Exception:
    _total_gb = 0

for pattern in ['codes_*.npy', 'fake.npy', 'fake.wav']:
    for f in glob.glob(os.path.join(REPO_DIR, pattern)):
        os.remove(f)
        print(f"   🧹 Removed: {os.path.basename(f)}")

gc.collect()
print()

# ════════════════════════════════════════════════════════════
# DASHBOARD RENDERER
# ════════════════════════════════════════════════════════════
def _get_ram():
    try:
        with open('/proc/meminfo') as f:
            mi = {l.split(':')[0]: l.split(':')[1].strip() for l in f}
        used  = (int(mi['MemTotal'].split()[0]) - int(mi['MemAvailable'].split()[0])) / 1e6
        total = int(mi['MemTotal'].split()[0]) / 1e6
        return used, total
    except Exception:
        return 0, _total_gb or 96

def _bar(pct, w=26, fill='█', empty='░', color='#00d4ff'):
    n = int(w * max(0, min(100, pct)) / 100)
    b = fill * n + empty * (w - n)
    return (
        '<span style="color:' + color +
        ';font-family:monospace;letter-spacing:1px">' + b + '</span>'
    )

def render_dashboard(stage, stage_label, log_lines, elapsed,
                     tok_done=0, tok_est=0, stage_pct=0, overall_pct=0,
                     status='RUNNING', error_msg=None,
                     cur_tps=0.0, cur_eta=0.0):
    ram_used, ram_total = _get_ram()
    ram_pct  = (ram_used / ram_total * 100) if ram_total else 0
    mins, secs = divmod(int(elapsed), 60)
    elapsed_str = f"{mins:02d}:{secs:02d}"

    # Speed / ETA — prefer live values from tqdm parse
    if cur_tps > 0:
        tps_str = f"{cur_tps:.2f} tok/s"
        if cur_eta > 0:
            em, es = divmod(int(cur_eta), 60)
            eta_str = f"~{em:02d}:{es:02d}"
        elif tok_est > 0 and tok_done >= tok_est:
            eta_str = "almost done"
        else:
            eta_str = "─"
    elif tok_done > 0 and elapsed > 2 and tok_est > tok_done:
        tps = tok_done / elapsed
        eta_sec = (tok_est - tok_done) / tps
        tps_str = f"{tps:.2f} tok/s"
        em, es = divmod(int(eta_sec), 60)
        eta_str = f"~{em:02d}:{es:02d}"
    elif tok_done > 0 and elapsed > 2:
        tps = tok_done / elapsed
        tps_str = f"{tps:.2f} tok/s"
        eta_str = "almost done"
    else:
        tps_str = eta_str = "─"

    sc = {'RUNNING': '#00d4ff', 'DONE': '#7fff00',
          'ERROR': '#ff4444', 'WAITING': '#ffa500'}
    status_color = sc.get(status, '#00d4ff')

    log_html = ""
    for line in log_lines[-9:]:
        safe = _html.escape(str(line))
        lo = line.lower()
        if any(w in lo for w in ['error', 'exception', 'failed', 'traceback']):
            c = '#ff6868'
        elif any(w in lo for w in ['info', 'load', 'done', 'complete', 'saved']):
            c = '#88ff88'
        elif any(w in lo for w in ['warn', 'warning', 'skip']):
            c = '#ffbb44'
        elif line.startswith('[') and ']' in line:
            c = '#aaddff'
        else:
            c = '#7aa8cc'
        log_html += (
            '<div style="color:' + c +
            ';margin:1px 0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
            '&gt; ' + safe + '</div>'
        )

    err_block = ""
    if error_msg:
        err_block = (
            '<div style="margin-top:8px;padding:6px 10px;background:#3a0000;'
            'border:1px solid #ff4444;border-radius:4px">'
            '<b style="color:#ff6666">⚠ ERROR:</b>'
            '<span style="color:#ffaaaa"> ' +
            _html.escape(str(error_msg)[:220]) +
            '</span></div>'
        )

    tok_disp   = f"{tok_done:,}" if tok_done else "─"
    total_disp = f"~{tok_est:,}" if tok_est  else "─"

    sp_str  = f"{stage_pct:.1f}%"
    op_str  = f"{overall_pct:.1f}%"
    ru_str  = f"{ram_used:.1f}/{ram_total:.0f} GB"

    html = (
        '<div style="background:#080c18;border:2px solid #00d4ff;border-radius:10px;'
        'padding:18px 22px;font-family:\'Courier New\',monospace;'
        'box-shadow:0 0 30px rgba(0,212,255,0.25);margin:12px 0;max-width:820px">'

        # header
        '<div style="display:flex;justify-content:space-between;align-items:flex-start;'
        'border-bottom:1px solid #162840;padding-bottom:10px;margin-bottom:14px">'
        '<div>'
        '<div style="color:#00d4ff;font-size:17px;font-weight:bold;letter-spacing:2px">'
        '🤖 S2-PRO SYNTHESIS ENGINE</div>'
        '<div style="color:#3a6080;font-size:11px;margin-top:3px">'
        'fishaudio/s2-pro &nbsp;·&nbsp; TPU v5e-1 HOST RAM MODE &nbsp;·&nbsp; CPU INFERENCE'
        '</div></div>'
        '<div style="text-align:right">'
        '<div style="color:' + status_color + ';font-weight:bold;font-size:14px;'
        'text-shadow:0 0 10px ' + status_color + '">● ' + status + '</div>'
        '<div style="color:#3a6080;font-size:11px;margin-top:3px">⏱ ' + elapsed_str + '</div>'
        '</div></div>'

        # stage bar
        '<div style="margin-bottom:14px">'
        '<div style="color:#3a6080;font-size:10px;letter-spacing:2px;margin-bottom:5px">'
        'STAGE ' + str(stage) + ' / 3</div>'
        '<div style="color:#e8f4ff;font-size:13px;font-weight:bold;margin-bottom:7px">'
        + _html.escape(str(stage_label)) +
        '</div>'
        '<div style="display:flex;align-items:center;gap:12px">'
        + _bar(stage_pct, w=30, color='#00d4ff') +
        '<span style="color:#00d4ff;font-size:13px;min-width:48px">' + sp_str + '</span>'
        '</div></div>'

        # metrics grid
        '<div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:8px;margin-bottom:14px">'

        '<div style="background:#0c1525;border:1px solid #162840;border-radius:5px;padding:9px 10px">'
        '<div style="color:#3a6080;font-size:9px;letter-spacing:1px;margin-bottom:3px">TOKENS GEN</div>'
        '<div style="color:#7fff00;font-size:17px;font-weight:bold">' + tok_disp + '</div>'
        '<div style="color:#3a6080;font-size:9px">of ' + total_disp + '</div>'
        '</div>'

        '<div style="background:#0c1525;border:1px solid #162840;border-radius:5px;padding:9px 10px">'
        '<div style="color:#3a6080;font-size:9px;letter-spacing:1px;margin-bottom:3px">SPEED</div>'
        '<div style="color:#ff9f43;font-size:17px;font-weight:bold">' + tps_str + '</div>'
        '<div style="color:#3a6080;font-size:9px">ETA ' + eta_str + '</div>'
        '</div>'

        '<div style="background:#0c1525;border:1px solid #162840;border-radius:5px;padding:9px 10px">'
        '<div style="color:#3a6080;font-size:9px;letter-spacing:1px;margin-bottom:3px">OVERALL</div>'
        '<div style="margin:4px 0">' + _bar(overall_pct, w=12, fill='▓', empty='░', color='#7fff00') + '</div>'
        '<div style="color:#7fff00;font-size:11px">' + op_str + '</div>'
        '</div>'

        '<div style="background:#0c1525;border:1px solid #162840;border-radius:5px;padding:9px 10px">'
        '<div style="color:#3a6080;font-size:9px;letter-spacing:1px;margin-bottom:3px">HOST RAM</div>'
        '<div style="color:#ff6b35;font-size:12px;font-weight:bold">' + ru_str + '</div>'
        '<div style="margin-top:3px">' + _bar(ram_pct, w=12, fill='▓', empty='░', color='#ff6b35') + '</div>'
        '</div>'
        '</div>'

        # log
        '<div style="background:#050912;border:1px solid #162840;border-radius:5px;padding:10px 12px">'
        '<div style="color:#3a6080;font-size:9px;letter-spacing:2px;margin-bottom:7px">'
        '◈ LIVE LOG — LAST 9 LINES</div>'
        '<div style="font-size:11px;line-height:1.65;overflow:hidden">' + log_html + '</div>'
        '</div>'

        + err_block +

        '<div style="margin-top:10px;text-align:center;color:#162840;font-size:9px;letter-spacing:2px">'
        '─── JARVIS SYSTEMS ONLINE ─── fishaudio/s2-pro ─── HOST RAM MODE ───'
        '</div>'
        '</div>'
    )
    return html

# ── Create dashboard display handle ──────────────────────────
dash = display(HTML(render_dashboard(0, 'Initialising pipeline...', [], 0)),
               display_id=True)

def upd(**kw):
    try:
        dash.update(HTML(render_dashboard(**kw)))
    except Exception:
        pass

# ════════════════════════════════════════════════════════════
# REAL-PROGRESS PARSER  (reads tqdm + fish-speech log lines)
# ════════════════════════════════════════════════════════════
# tqdm full:  "  25%|████| 128/512 [00:30<01:30, 1.42it/s]"
_RE_TQDM = re.compile(
    r'(\d+)/(\d+)\s*\[[\d:]+<([\d:?]+),\s*([\d.]+)\s*(?:it|tok)/s'
)
# bare X/Y:    "128/512" anywhere in line
_RE_XY   = re.compile(r'\b(\d+)\s*/\s*(\d+)\b')
# fish-speech: "generate 512 tokens"
_RE_GEN  = re.compile(r'generat\w*\s+(\d+)\s*tokens?', re.IGNORECASE)
# key=value:   "tokens: 512" / "token_count=512"
_RE_TOKV = re.compile(r'token\w*\s*[=:]\s*(\d+)', re.IGNORECASE)
# speed hint:  "2.34 tok/s" / "2.34it/s"
_RE_TPS  = re.compile(r'([\d.]+)\s*(?:tok|it)/s', re.IGNORECASE)

def _parse_eta_str(s):
    """Convert '01:30' or '1:02:30' to seconds float. Returns 0 on failure."""
    try:
        if s in ('?', ''):
            return 0.0
        parts = [int(x) for x in s.split(':')]
        if len(parts) == 2:
            return parts[0] * 60 + parts[1]
        if len(parts) == 3:
            return parts[0] * 3600 + parts[1] * 60 + parts[2]
    except Exception:
        pass
    return 0.0

def _parse_progress(line, tok_done, tok_total, cur_tps):
    """Return updated (tok_done, tok_total, cur_tps, cur_eta)."""
    cur_eta = 0.0

    # 1. Full tqdm — most accurate, includes ETA
    m = _RE_TQDM.search(line)
    if m:
        td   = int(m.group(1))
        tt   = int(m.group(2))
        eta  = _parse_eta_str(m.group(3))
        tps  = float(m.group(4))
        return max(tok_done, td), max(tok_total, tt), tps, eta

    # 2. Bare "X/Y" fraction (tqdm without timer, custom progress)
    for m in _RE_XY.finditer(line):
        td, tt = int(m.group(1)), int(m.group(2))
        if tt > 10:
            tok_done  = max(tok_done, td)
            tok_total = max(tok_total, tt)
            break

    # 3. "generate N tokens"
    m = _RE_GEN.search(line)
    if m:
        td = int(m.group(1))
        tok_done  = max(tok_done, td)
        tok_total = max(tok_total, td)

    # 4. "tokens: N"
    m = _RE_TOKV.search(line)
    if m:
        tok_done = max(tok_done, int(m.group(1)))

    # 5. Standalone speed hint
    m = _RE_TPS.search(line)
    if m:
        cur_tps = float(m.group(1))
        if cur_tps > 0 and tok_total > tok_done:
            cur_eta = (tok_total - tok_done) / cur_tps

    return tok_done, tok_total, cur_tps, cur_eta

# ════════════════════════════════════════════════════════════
# SUBPROCESS RUNNER  (real progress — no fake line-count cap)
# ════════════════════════════════════════════════════════════
def run_stage(cmd_str, stage_num, stage_label, log_lines,
              t0_global, pct_start=0, pct_end=100, est_tokens=0):
    env = os.environ.copy()
    env.update({
        'CUDA_VISIBLE_DEVICES': '',
        'OMP_NUM_THREADS':      '8',
        'MKL_NUM_THREADS':      '8',
        'ACCELERATE_USE_CPU':   'true',
        'HF_HUB_OFFLINE':       '1',
        'PYTHONPATH':           ':'.join([REPO_DIR] + sys.path),
        'PYTHONUNBUFFERED':     '1',
    })
    ts = time.strftime('%H:%M:%S')
    log_lines.append(f"[{ts}] ▶ Stage {stage_num}: {stage_label}")
    log_lines.append(f"[{ts}] CMD: {cmd_str[:100]}")

    proc = subprocess.Popen(
        cmd_str, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env, cwd=REPO_DIR
    )

    tok_done  = 0
    tok_total = est_tokens   # refined by real output as it arrives
    cur_tps   = 0.0
    cur_eta   = 0.0
    line_no   = 0
    last_upd  = time.time()
    sp        = 0.0

    for raw in proc.stdout:
        line = raw.rstrip()
        line_no += 1
        if line.strip():
            log_lines.append(line)

        # Parse real progress from this line
        tok_done, tok_total, cur_tps, cur_eta = _parse_progress(
            line, tok_done, tok_total, cur_tps
        )

        # ── Compute stage_pct from REAL data ─────────────────
        if tok_total > 0 and tok_done > 0:
            # Real token counts available — direct ratio, cap at 99.5 until done
            sp = min(99.5, tok_done / tok_total * 100)
        else:
            # Still in loading phase — slow climb, never exceeds 28
            sp = min(28.0, line_no * 0.2)

        op = pct_start + (pct_end - pct_start) * sp / 100

        if time.time() - last_upd > 0.8:
            upd(stage=stage_num, stage_label=stage_label,
                log_lines=log_lines,
                elapsed=time.time() - t0_global,
                tok_done=tok_done, tok_est=tok_total,
                stage_pct=sp, overall_pct=op,
                status='RUNNING',
                cur_tps=cur_tps, cur_eta=cur_eta)
            last_upd = time.time()

    proc.wait()
    rc      = proc.returncode
    elapsed = time.time() - t0_global

    if rc == 0:
        log_lines.append(
            f"[{time.strftime('%H:%M:%S')}] ✅ Stage {stage_num} complete  ({elapsed:.0f}s total)"
        )
        upd(stage=stage_num, stage_label=f"✅ {stage_label}",
            log_lines=log_lines, elapsed=elapsed,
            tok_done=tok_done, tok_est=tok_total,
            stage_pct=100, overall_pct=pct_end,
            status='DONE', cur_tps=0.0, cur_eta=0.0)
    else:
        log_lines.append(
            f"[{time.strftime('%H:%M:%S')}] ❌ Stage {stage_num} exit {rc}"
        )
        upd(stage=stage_num, stage_label=f"❌ {stage_label}",
            log_lines=log_lines, elapsed=elapsed,
            tok_done=tok_done, tok_est=tok_total,
            stage_pct=sp, overall_pct=op,
            status='ERROR', error_msg=f"Exit {rc} — check log above",
            cur_tps=0.0, cur_eta=0.0)
    return rc

# ════════════════════════════════════════════════════════════
# STAGE 1 (optional): encode reference audio
# ════════════════════════════════════════════════════════════
t0         = time.time()
log_lines  = []
prompt_npy = ''

if REFERENCE_AUDIO and os.path.exists(REFERENCE_AUDIO):
    log_lines.append(
        f"[{time.strftime('%H:%M:%S')}] Reference: {os.path.basename(REFERENCE_AUDIO)}"
    )
    cmd_enc = (
        f"{sys.executable} -m fish_speech.models.dac.inference"
        f" -i {shlex.quote(REFERENCE_AUDIO)}"
        f" --checkpoint-path {shlex.quote(os.path.join(MODEL_DIR, 'codec.pth'))}"
        f" --output-path {shlex.quote(os.path.join(OUTPUT_DIR, 'ref_prompt.npy'))}"
        f" --device cpu"
    )
    rc1 = run_stage(cmd_enc, 1, "DAC Encode — Reference Audio → Tokens",
                    log_lines, t0, pct_start=0, pct_end=8, est_tokens=150)
    if rc1 == 0:
        cands = (sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npy')))
               + sorted(glob.glob(os.path.join(REPO_DIR,   'fake.npy'))))
        if cands:
            prompt_npy = cands[-1]
            log_lines.append(
                f"[{time.strftime('%H:%M:%S')}] Prompt tokens: {prompt_npy}"
            )
        else:
            log_lines.append(
                f"[{time.strftime('%H:%M:%S')}] ⚠️  No .npy — proceeding without voice clone"
            )
    else:
        log_lines.append(
            f"[{time.strftime('%H:%M:%S')}] ⚠️  Stage 1 failed — proceeding without voice clone"
        )
else:
    log_lines.append(f"[{time.strftime('%H:%M:%S')}] Stage 1 skipped (no reference audio)")
    upd(stage=1, stage_label="Stage 1: Skipped — no reference audio",
        log_lines=log_lines, elapsed=time.time() - t0,
        stage_pct=100, overall_pct=8, status='WAITING',
        cur_tps=0.0, cur_eta=0.0)
    time.sleep(0.4)

# ════════════════════════════════════════════════════════════
# HINGLISH TEXT NORMALIZER  (audiobook-optimized)
# ════════════════════════════════════════════════════════════
import re

def normalize_hinglish(text, narrator_tag="", audiobook_mode=True):
    """
    Prepares Hinglish text for best S2-Pro output:
      1. Phonetically expands common English abbreviations & tech terms
         that the model mispronounces when written as abbreviations.
      2. In AUDIOBOOK_MODE: strips all existing [] tags from input,
         and prepends a single well-crafted NARRATOR_TAG.
         (Over-stacked tags fight each other and degrade quality.)
      3. Cleans up spacing/punctuation.

    Why strip user tags in AUDIOBOOK_MODE?
      S2-Pro tag effectiveness: tags work best as one clear instruction.
      Stacking [deep female voice][loud][excited][shocked] in one sentence
      causes the model to blend/cancel them, often reverting to a neutral
      default voice with artifacts. A single, descriptive opening tag is
      consistently better for narration/audiobooks.
    """
    if not audiobook_mode:
        return text.strip()

    # ── Step 1: strip all existing [] tags from input ────────
    cleaned = re.sub(r'\[([^\]]+)\]', '', text)

    # ── Step 2: phonetic expansion for common tech/English terms ──
    # Order matters — longer patterns first to avoid partial matches
    EXPANSIONS = [
        # Tech & AI terms that the model garbles in Hinglish context
        (r'\bS-?2\b',       'S-Two'),
        (r'\bS-?1\b',       'S-One'),
        (r'\bGPT-?4\b',     'G-P-T-Four'),
        (r'\bGPT-?3\b',     'G-P-T-Three'),
        (r'\bGPT\b',        'G-P-T'),
        (r'\bChatGPT\b',    'Chat-G-P-T'),
        (r'\bLLM\b',        'L-L-M'),
        (r'\bAPI\b',        'A-P-I'),
        (r'\bURL\b',        'U-R-L'),
        (r'\bUI\b',         'U-I'),
        (r'\bAI\b',         'A-I'),
        (r'\bML\b',         'M-L'),
        (r'\bDL\b',         'D-L'),
        (r'\bNLP\b',        'N-L-P'),
        (r'\bCNN\b',        'C-N-N'),
        (r'\bRNN\b',        'R-N-N'),
        (r'\bTTS\b',        'T-T-S'),
        (r'\bSTT\b',        'S-T-T'),
        (r'\bOCR\b',        'O-C-R'),
        (r'\bCPU\b',        'C-P-U'),
        (r'\bGPU\b',        'G-P-U'),
        (r'\bTPU\b',        'T-P-U'),
        (r'\bRAM\b',        'R-A-M'),
        (r'\bROI\b',        'R-O-I'),
        (r'\bCEO\b',        'C-E-O'),
        (r'\bCTO\b',        'C-T-O'),
        (r'\bPM\b',         'P-M'),
        (r'\bUSB\b',        'U-S-B'),
        (r'\bSMS\b',        'S-M-S'),
        (r'\bOTP\b',        'O-T-P'),
        (r'\bUPI\b',        'U-P-I'),
        (r'\bEMI\b',        'E-M-I'),
        # English words that look like abbrevs but should be kept
        # (none needed here — we only expand all-caps or known patterns)
    ]
    for pattern, replacement in EXPANSIONS:
        cleaned = re.sub(pattern, replacement, cleaned)

    # ── Step 3: fix dash-spacing around em-dashes / hyphens ──
    cleaned = re.sub(r'\s*—\s*', ' — ', cleaned)
    cleaned = re.sub(r'\s*-\s*', '-', cleaned)

    # ── Step 4: collapse extra whitespace ────────────────────
    cleaned = re.sub(r'[ \t]+', ' ', cleaned).strip()

    # ── Step 5: prepend single narrator tag ──────────────────
    if narrator_tag.strip():
        cleaned = f"[{narrator_tag.strip()}] {cleaned}"

    return cleaned

# ════════════════════════════════════════════════════════════
# STAGE 2: text → semantic tokens
# ════════════════════════════════════════════════════════════
# Apply Hinglish normalizer — expands abbreviations + injects NARRATOR_TAG
raw_text   = TEXT_TO_SYNTH.replace('\n', ' ').strip()
text_clean = normalize_hinglish(raw_text,
                                narrator_tag=NARRATOR_TAG if AUDIOBOOK_MODE else '',
                                audiobook_mode=AUDIOBOOK_MODE)
word_count = len(text_clean.split())
est_tokens = max(word_count * 25, 500)   # rough: ~25 codec tokens per word

log_lines.append(f"[{time.strftime('%H:%M:%S')}] Stage 2 starting: {word_count} words, est ~{est_tokens} tokens")
if AUDIOBOOK_MODE:
    log_lines.append(f"[{time.strftime('%H:%M:%S')}] Normalized text: {text_clean[:120]}...")
    print(f"\n📝 NORMALIZED TEXT (first 200 chars):\n   {text_clean[:200]}")

cmd_t2s = (
    f"{sys.executable} -m fish_speech.models.text2semantic.inference"
    f" --text {shlex.quote(text_clean)}"
    f" --checkpoint-path {shlex.quote(MODEL_DIR)}"
    f" --output-dir {shlex.quote(OUTPUT_DIR)}"
    f" --device cpu"
    f" --temperature {TEMPERATURE}"
    f" --top-p {TOP_P}"
    f" --top-k {int(TOP_K)}"
    f" --num-samples {NUM_SAMPLES}"
)
if MAX_NEW_TOKENS > 0:
    cmd_t2s += f" --max-new-tokens {int(MAX_NEW_TOKENS)}"
if prompt_npy:
    cmd_t2s += f" --prompt-tokens {shlex.quote(prompt_npy)}"
    if PROMPT_TEXT.strip():
        cmd_t2s += f" --prompt-text {shlex.quote(PROMPT_TEXT)}"
if COMPILE:
    cmd_t2s += " --compile"

rc2 = run_stage(cmd_t2s, 2,
                "Text → Semantic Tokens  (4B LLM — the slow stage on CPU)",
                log_lines, t0, pct_start=8, pct_end=85, est_tokens=est_tokens)

# ════════════════════════════════════════════════════════════
# STAGE 3: semantic tokens → WAV
# ════════════════════════════════════════════════════════════
codes = (sorted(glob.glob(os.path.join(OUTPUT_DIR, 'codes_*.npy')))
       + sorted(glob.glob(os.path.join(REPO_DIR,   'codes_*.npy'))))

if not codes:
    upd(stage=2, stage_label="❌ codes_*.npy not found after Stage 2",
        log_lines=log_lines, elapsed=time.time() - t0,
        stage_pct=100, overall_pct=85, status='ERROR',
        error_msg="Stage 2 may have failed silently. Check log.",
        cur_tps=0.0, cur_eta=0.0)
    raise FileNotFoundError("codes_*.npy not found.")

log_lines.append(f"[{time.strftime('%H:%M:%S')}] Found {len(codes)} code file(s) to decode")
OUTPUT_WAV_PATHS = []  # will hold path for each sample
pct_per_sample = (100 - 85) / max(len(codes), 1)

for ci, codes_file in enumerate(codes):
    sz_kb = os.path.getsize(codes_file) / 1024
    out_wav = os.path.join(OUTPUT_DIR, f'output_sample_{ci}.wav')
    log_lines.append(
        f"[{time.strftime('%H:%M:%S')}] Decoding sample {ci}: {os.path.basename(codes_file)} ({sz_kb:.1f} KB)"
    )
    cmd_dec = (
        f"{sys.executable} -m fish_speech.models.dac.inference"
        f" -i {shlex.quote(codes_file)}"
        f" --checkpoint-path {shlex.quote(os.path.join(MODEL_DIR, 'codec.pth'))}"
        f" --output-path {shlex.quote(out_wav)}"
        f" --device cpu"
    )
    pct_s = 85 + ci * pct_per_sample
    pct_e = 85 + (ci + 1) * pct_per_sample
    rc3 = run_stage(cmd_dec, 3,
                    f"DAC Decode sample {ci+1}/{len(codes)} → WAV",
                    log_lines, t0, pct_start=pct_s, pct_end=pct_e, est_tokens=400)
    if rc3 == 0 and os.path.exists(out_wav):
        OUTPUT_WAV_PATHS.append(out_wav)

# ════════════════════════════════════════════════════════════
# RESULT
# ════════════════════════════════════════════════════════════
total_elapsed = time.time() - t0

# Collect all decoded sample WAVs (output_sample_0.wav, output_sample_1.wav, ...)
if not OUTPUT_WAV_PATHS:
    OUTPUT_WAV_PATHS = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'output_sample_*.wav')))

# Fallback: pick up any stray fake.wav from fish-speech repo dir
if not OUTPUT_WAV_PATHS:
    fb = sorted(glob.glob(os.path.join(REPO_DIR, 'fake.wav')))
    OUTPUT_WAV_PATHS = fb

if OUTPUT_WAV_PATHS:
    import shutil as _shutil, soundfile as _sf

    # Rename sample_0 → output_s2pro.wav for backward compat (Cell 8 expects it)
    OUTPUT_WAV_PATH = os.path.join(OUTPUT_DIR, 'output_s2pro.wav')
    if OUTPUT_WAV_PATHS[0] != OUTPUT_WAV_PATH:
        _shutil.copy2(OUTPUT_WAV_PATHS[0], OUTPUT_WAV_PATH)

    print(f"\n{SEP}")
    print(f"  🎉 SYNTHESIS COMPLETE!  ({len(OUTPUT_WAV_PATHS)} sample(s))")
    print(f"{'─'*62}")
    for idx, wp in enumerate(OUTPUT_WAV_PATHS):
        try:
            data, sr = _sf.read(wp)
            dur = len(data) / sr
        except Exception:
            dur, sr = 0, 44100
        wav_mb = os.path.getsize(wp) / 1e6
        marker = " ← Cell 8 plays this" if idx == 0 else ""
        print(f"  Sample {idx}      : {os.path.basename(wp)}  ({dur:.1f}s · {wav_mb:.2f} MB){marker}")
    print(f"{'─'*62}")
    print(f"  Total time   : {total_elapsed:.0f}s  ({total_elapsed/60:.1f} min)")
    print(f"{SEP}")
    print(f"\n  💡 TIP: If Sample 0 voice isn't right, compare with Sample 1 (run Cell 8")
    print(f"          and change OUTPUT_WAV_PATH to inference_outputs/output_sample_1.wav).")
    print(f"  💡 To lock in a good voice: upload that sample as reference audio in Cell 4.")

    try:
        data0, sr0 = _sf.read(OUTPUT_WAV_PATH)
        dur0 = len(data0) / sr0
        wav_mb0 = os.path.getsize(OUTPUT_WAV_PATH) / 1e6
    except Exception:
        dur0, sr0, wav_mb0 = 0, 44100, 0

    log_lines.append(
        f"[{time.strftime('%H:%M:%S')}] ✅ {len(OUTPUT_WAV_PATHS)} sample(s) ready"
    )
    log_lines.append(f"[{time.strftime('%H:%M:%S')}] Total time: {total_elapsed:.0f}s")

    upd(stage=3,
        stage_label=(
            f"✅ COMPLETE — {len(OUTPUT_WAV_PATHS)} sample(s) · {dur0:.1f}s · {total_elapsed:.0f}s total"
        ),
        log_lines=log_lines, elapsed=total_elapsed,
        tok_done=est_tokens, tok_est=est_tokens,
        stage_pct=100, overall_pct=100, status='DONE',
        cur_tps=0.0, cur_eta=0.0)

else:
    OUTPUT_WAV_PATH  = ''
    OUTPUT_WAV_PATHS = []
    upd(stage=3, stage_label="❌ No WAV found after Stage 3",
        log_lines=log_lines, elapsed=total_elapsed,
        stage_pct=99, overall_pct=99, status='ERROR',
        error_msg="Pipeline finished but no .wav produced. Check log.",
        cur_tps=0.0, cur_eta=0.0)
    raise RuntimeError("No WAV output found. Check dashboard log above.")



══════════════════════════════════════════════════════════════
  CELL 7 — S2-PRO INFERENCE
══════════════════════════════════════════════════════════════

🔍 PRE-FLIGHT CHECKS
   TEXT_TO_SYNTH   : 142 chars / 21 words
   REFERENCE_AUDIO : 'uploads/Fiction___1.0x.mp3'
   MODEL_DIR       : /content/checkpoints/s2-pro
   OUTPUT_DIR      : /content/inference_outputs
   TEMPERATURE     : 0.5
   TOP_P / TOP_K   : 0.8 / 50
   COMPILE         : False
   ✅ config.json
   ✅ codec.pth
   ✅ Input text present
   RAM available   : 46.7 GB / 49.3 GB




📝 NORMALIZED TEXT (first 200 chars):
   [young Indian woman, casual conversational Hinglish, warm and clear, natural Hindi-English mix, moderate pace, audiobook narration, urban Indian accent] अरे यार, यह S-Two model सुनो — A-I कमाल है! बिल


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — PLAY & DOWNLOAD OUTPUT AUDIO  (multi-sample aware)
# ════════════════════════════════════════════════════════════
import os, glob
from IPython.display import Audio, display, HTML
from google.colab import files

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 8 — PLAY & DOWNLOAD OUTPUT")
print(f"{SEP}\n")

# ── Collect all generated samples ─────────────────────────────
all_wavs = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'output_sample_*.wav')))
if not all_wavs:
    # Fallback: old single-file output
    cands = (sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.wav')))
           + sorted(glob.glob('/content/fish-speech/fake.wav')))
    all_wavs = cands

if not all_wavs:
    raise FileNotFoundError("No output WAV found. Run Cell 7 first.")

print(f"  Found {len(all_wavs)} sample(s):")
print()

import soundfile as _sf

for idx, wp in enumerate(all_wavs):
    if not os.path.exists(wp):
        continue
    wav_mb = os.path.getsize(wp) / 1e6
    try:
        data, sr = _sf.read(wp)
        duration = len(data) / sr
        channels = 'Stereo' if data.ndim > 1 else 'Mono'
    except Exception as e:
        data, sr, duration, channels = None, 44100, 0, 'Unknown'
        print(f"   ⚠️  Could not read audio metadata: {e}")

    display(HTML(f"""
<div style="background:#080c18;border:1px solid #00d4ff;border-radius:8px;
            padding:12px 16px;font-family:monospace;color:#e8f4ff;
            margin:8px 0;max-width:700px">
  <span style="color:#00d4ff;font-weight:bold;font-size:14px">
    🎙 Sample {idx}
  </span>
  &nbsp;&nbsp;
  <span style="color:#3a6080;font-size:11px">
    {os.path.basename(wp)} &nbsp;·&nbsp; {duration:.1f}s &nbsp;·&nbsp;
    {wav_mb:.2f} MB &nbsp;·&nbsp; {sr} Hz &nbsp;·&nbsp; {channels}
  </span>
  {"<br><span style='color:#7fff00;font-size:10px'>  ← default (output_s2pro.wav)</span>" if idx == 0 else ""}
</div>"""))

    display(Audio(wp, autoplay=False))
    print()

# ── Waveform plot for sample 0 ─────────────────────────────────
try:
    import matplotlib, numpy as np
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    wp0 = all_wavs[0]
    data0, sr0 = _sf.read(wp0)
    audio_mono = data0[:,0] if data0.ndim > 1 else data0
    duration0  = len(data0) / sr0

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 5))
    fig.patch.set_facecolor('#080c18')
    for ax in (ax1, ax2):
        ax.set_facecolor('#0c1525')
        for sp in ax.spines.values():
            sp.set_color('#162840')
        ax.tick_params(colors='#3a6080', labelsize=8)

    t = np.linspace(0, duration0, len(audio_mono))
    ax1.plot(t, audio_mono, lw=0.35, color='#00d4ff', alpha=0.9)
    ax1.set_ylabel('Amplitude', color='#3a6080', fontsize=9)
    ax1.set_title(f'Waveform — Sample 0  ·  {sr0} Hz  ·  {duration0:.1f}s',
                  color='#00d4ff', fontsize=10, pad=6)
    ax1.set_xlim(0, duration0)
    ax1.grid(True, alpha=0.12, color='#162840')

    spec = audio_mono[:min(len(audio_mono), sr0*45)]
    ax2.specgram(spec, Fs=sr0, cmap='plasma', NFFT=1024, noverlap=512)
    ax2.set_ylabel('Freq (Hz)', color='#3a6080', fontsize=9)
    ax2.set_xlabel('Time (s)', color='#3a6080', fontsize=9)
    ax2.set_title('Spectrogram', color='#ff9f43', fontsize=10, pad=6)
    ax2.set_ylim(0, min(8000, sr0 // 2))

    fig.suptitle('🐟 fishaudio/s2-pro  ─  Generated Audio',
                 color='#7fff00', fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    outpng = os.path.join(OUTPUT_DIR, 'waveform_sample0.png')
    plt.savefig(outpng, dpi=130, bbox_inches='tight', facecolor='#080c18')
    plt.show()
    print("   ✅ Waveform + spectrogram for Sample 0 plotted above.")
except Exception as e:
    print(f"   ⚠️  Plot failed: {e}")

# ── Download all samples ────────────────────────────────────────
print(f"\n💾 Downloading {len(all_wavs)} sample(s)...")
for wp in all_wavs:
    if os.path.exists(wp):
        files.download(wp)
        print(f"   ↓ {os.path.basename(wp)}")

# ── Voice lock-in instructions ─────────────────────────────────
print(f"""
{SEP}
  🎉 DONE!

  ┌─────────────────────────────────────────────────────────────┐
  │  HOW TO GET A CONSISTENT INDIAN FEMALE VOICE                │
  │                                                             │
  │  Tags like [young female voice] control STYLE + PROSODY     │
  │  only — they cannot change the fundamental speaker timbre.  │
  │  Each run without reference audio picks a new random voice. │
  │                                                             │
  │  To lock in a specific voice:                               │
  │  1. Listen to Sample 0 and Sample 1 above                   │
  │  2. Pick the one that sounds most like what you want        │
  │  3. Re-run Cell 4 → upload that sample as reference audio   │
  │  4. Set PROMPT_TEXT in Cell 2 = exact transcript of text    │
  │  5. Now every run will clone that voice consistently        │
  │                                                             │
  │  Best reference: a clean 10-30s Indian female voice clip    │
  │  (no music, no reverb, clear pronunciation)                 │
  └─────────────────────────────────────────────────────────────┘
{SEP}""")
